# AE Evaluation (Autoencoder)

<div style="text-align: justify">

The following notebook evaluates the trained <b>Autoencoder (AE)</b> for the <b>Tau Anomaly Detection</b> analysis. It loads the trained checkpoint, computes per-event anomaly scores, and produces evaluation metrics (ROC AUC, SIC) and diagnostic visualisations including reconstruction quality, per-feature importance, and latent space analysis.

</div>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis and model configuration |
| DataModule | `datamodule.AnomalyDataModule` | Read mc.parquet, split bkg/sig, fit scaler, build dataloaders |
| Load | `ae.Autoencoder.load_from_checkpoint` | Load trained AE checkpoint |
| Scores | `anomaly.reconstruction_error` | Compute per-event anomaly scores on predict set |
| Threshold | `anomaly.compute_threshold` | Percentile-based anomaly threshold on background |
| Metrics | `evaluation.compute_metrics` | ROC AUC, SIC, per-origin ROC |
| Plots | `plots` | Reconstruction error, ROC, SIC, per-feature importance, latent space |
| Save | `io.save_dataframe`, `json.dump` | Persist scores parquet and metrics JSON |

The same pipeline is available as a CLI via `uv run python run.py stage=evaluate model=ae`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Machine Learning:
* [PyTorch](https://pytorch.org/)
* [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/)
* [scikit-learn](https://scikit-learn.org/)

Visualization:
* [matplotlib](https://matplotlib.org/)

### Notebook

Activating autoreload of imported modules.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

Initializing the project root.

In [ ]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [ ]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration with the AE model config. All path parameters must match those used during training.

In [ ]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config", overrides=["model=ae"])

Resolving input and output directories from config.

In [ ]:
from src.processing.analysis import get_output_paths

output_paths = get_output_paths(cfg)
dataframes_dir = path / output_paths["dataframes_dir"]
models_dir = path / output_paths["models_dir"]
plots_dir = path / output_paths["plots_dir"] / "ae_evaluation"

plots_dir.mkdir(parents=True, exist_ok=True)

## DataModule

Setting up the `AnomalyDataModule`. It reads the MC parquet, separates background from signal, fits a scaler on the background training split, and builds the predict set (background validation + all signal).

In [ ]:
import lightning as L

L.seed_everything(cfg.seed, workers=True)

In [ ]:
from src.models.datamodule import AnomalyDataModule

background_origins = {
    s["id"]
    for s in cfg.samples.background.samples
    if s["id"] not in set(cfg.samples.background.get("exclude", []))
}
print(f"Background origins: {background_origins}")

dm = AnomalyDataModule(
    mc_path=str(dataframes_dir / "mc.parquet"),
    background_origins=background_origins,
    normalization=cfg.model.normalization,
    val_fraction=cfg.pipeline.val_fraction,
    batch_size=cfg.model.batch_size,
    seed=cfg.seed,
)
dm.setup()
print(f"Features: {dm.n_features}")
print(f"Feature names: {dm.feature_names_}")

Gathering original features from the predict set.

In [ ]:
import torch

x_orig = torch.cat([batch[0] for batch in dm.predict_dataloader()])
print(
    f"Predict set: {len(x_orig):,} events "
    f"({int((dm.predict_labels == 0).sum()):,} bkg, "
    f"{int((dm.predict_labels == 1).sum()):,} sig)"
)

## Deserialization

### Load Checkpoint

Loading the trained Autoencoder from checkpoint.

In [ ]:
from omegaconf import OmegaConf

from src.models.ae import Autoencoder
from src.models.config import AEConfig

ae_params = dict(OmegaConf.to_container(cfg.model, resolve=True))
ae_cfg = AEConfig(**ae_params)
model = Autoencoder.load_from_checkpoint(
    models_dir / "ae.ckpt", cfg=ae_cfg, n_features=dm.n_features
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Autoencoder: {n_params:,} parameters")

### Anomaly Scores

Computing per-event reconstruction error on the predict set.

In [ ]:
import numpy as np

from src.models.anomaly import (
    build_scores_frame,
    compute_threshold,
    per_feature_error,
    reconstruction_error,
)

x_hat_list = []
with torch.no_grad():
    for batch in dm.predict_dataloader():
        x, _w = batch
        x_hat = model(x)
        x_hat_list.append(x_hat)

x_hat = torch.cat(x_hat_list)
scores = reconstruction_error(x_orig, x_hat).numpy()

print(f"AE scores: {len(scores):,} events")
print(f"  Background mean: {scores[dm.predict_labels == 0].mean():.6f}")
print(f"  Signal mean:     {scores[dm.predict_labels == 1].mean():.6f}")

### Threshold

In [ ]:
bkg_scores = scores[dm.predict_labels == 0]
sig_scores = scores[dm.predict_labels == 1]

threshold = compute_threshold(
    bkg_scores,
    strategy=cfg.pipeline.threshold_strategy,
    percentile=cfg.pipeline.threshold_percentile,
)
print(f"Threshold ({cfg.pipeline.threshold_strategy}): {threshold:.6f}")
print(f"  Background flagged: {(bkg_scores > threshold).sum():,} / {len(bkg_scores):,}")
print(f"  Signal flagged:     {(sig_scores > threshold).sum():,} / {len(sig_scores):,}")

## Metrics

### Summary Metrics

ROC AUC, max SIC, and per-origin ROC AUC.

In [ ]:
from src.models.evaluation import compute_metrics, compute_roc_curve, compute_sic_curve

scores_df = build_scores_frame(scores, dm.predict_labels, dm.predict_origins)
metrics = compute_metrics(
    labels=dm.predict_labels,
    scores=scores,
    scores_df=scores_df,
)
metrics["threshold"] = threshold
metrics["n_background"] = int((dm.predict_labels == 0).sum())
metrics["n_signal"] = int((dm.predict_labels == 1).sum())

print(f"AE ROC AUC: {metrics["roc_auc"]:.4f}")
print(f"AE max SIC: {metrics["max_sic"]:.4f}")

### Reconstruction Error Distribution

Comparing the reconstruction error distributions of background and signal events.

In [ ]:
from src.models.plots import plot_reconstruction_error
from src.visualization.plots import save_figure

fig = plot_reconstruction_error(
    bkg_scores, sig_scores, threshold=threshold, title="AE Reconstruction Error"
)
save_figure(fig, plots_dir / "reconstruction_error.png")
fig.show()

### ROC Curve

In [ ]:
from src.models.plots import plot_roc_curve

fpr, tpr, _ = compute_roc_curve(dm.predict_labels, scores)
fig = plot_roc_curve(fpr, tpr, auc=metrics["roc_auc"], title="AE ROC Curve")
save_figure(fig, plots_dir / "roc_curve.png")
fig.show()

### SIC Curve

Significance Improvement Characteristic: signal efficiency divided by the square root of background efficiency.

In [ ]:
from src.models.plots import plot_sic_curve

sic = compute_sic_curve(fpr, tpr)
fig = plot_sic_curve(fpr, tpr, sic, title="AE Significance Improvement")
save_figure(fig, plots_dir / "sic_curve.png")
fig.show()

### Per-Origin ROC

ROC AUC computed separately for each signal mass point.

In [ ]:
from src.models.plots import plot_roc_per_origin

if metrics.get("roc_per_origin"):
    fig = plot_roc_per_origin(
        metrics["roc_per_origin"], title="AE ROC AUC per Signal Origin"
    )
    save_figure(fig, plots_dir / "roc_per_origin.png")
    fig.show()
    for origin, auc in sorted(metrics["roc_per_origin"].items()):
        print(f"  {origin}: {auc:.4f}")

### Per-Feature Importance

Mean squared reconstruction error per feature, identifying which features contribute most to the anomaly score.

In [ ]:
from src.models.plots import plot_per_feature_importance

feat_errors = per_feature_error(x_orig, x_hat).numpy()
mean_feat_errors = feat_errors.mean(axis=0)
fig = plot_per_feature_importance(
    mean_feat_errors, dm.feature_names_, title="AE Per-Feature Reconstruction Error"
)
save_figure(fig, plots_dir / "per_feature_importance.png")
fig.show()

## Reconstruction Diagnostics

### Single Event Reconstruction

Bar chart comparing original vs reconstructed feature values for a single event.

In [ ]:
from src.models.plots import plot_reconstruction_performance

fig = plot_reconstruction_performance(
    x_orig[0].numpy(),
    x_hat[0].numpy(),
    dm.feature_names_,
    event_idx=0,
    title="Single Event Reconstruction (AE)",
)
save_figure(fig, plots_dir / "single_event.png")
fig.show()

### Feature Histograms

Per-feature distributions of original vs reconstructed values across the predict set.

In [ ]:
from src.models.plots import plot_feature_histograms

fig = plot_feature_histograms(
    x_orig.numpy(),
    x_hat.numpy(),
    dm.feature_names_,
    title="Feature Distributions (AE)",
)
save_figure(fig, plots_dir / "feature_histograms.png")
fig.show()

## Latent Space

Visualising the encoder"s latent representations `z = encoder(x)` on the predict set (background + signal). The autoencoder was trained on background only, so any separation visible here emerges unsupervised.

In [ ]:
model.eval()
with torch.no_grad():
    z_np = model.encode(x_orig).numpy()

print(f"Latent space: {z_np.shape[0]:,} events, {z_np.shape[1]} dimensions")

### Latent Dimension Histograms

Distribution of the latent code per dimension, coloured by background vs signal.

In [ ]:
from src.models.plots import plot_latent_histograms

fig = plot_latent_histograms(z_np, labels=dm.predict_labels, title="Latent Dimension Histograms (AE)")
save_figure(fig, plots_dir / "latent_histograms.png")
fig.show()

### Latent Space 2D (t-SNE)

t-SNE projection of the latent space. Clear clusters suggest the autoencoder has learnt a structured representation separating background from signal.

In [ ]:
from sklearn.manifold import TSNE
from src.models.plots import plot_latent_space_2d

n_sample = min(5000, len(z_np))
rng = np.random.default_rng(42)
idx = rng.choice(len(z_np), n_sample, replace=False)
embedding = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(z_np[idx])

fig = plot_latent_space_2d(embedding, dm.predict_labels[idx], method="t-SNE", title="AE Latent Space (t-SNE)")
save_figure(fig, plots_dir / "tsne.png")
fig.show()

### Latent Pairplot

Pairwise scatter of the first latent dimensions, coloured by background vs signal.

In [ ]:
from src.models.plots import plot_latent_pairplot

fig = plot_latent_pairplot(z_np, labels=dm.predict_labels, title="AE Latent Pairplot")
save_figure(fig, plots_dir / "latent_pairplot.png")
fig.show()

## Serialization

### Scores DataFrame

Saving the anomaly scores to parquet.

In [ ]:
scores_path = dataframes_dir / "ae_scores.parquet"
scores_df.to_parquet(scores_path)
print(f"Saved scores: {scores_path}")
scores_df.head()

### Metrics JSON

Saving evaluation metrics.

In [ ]:
import json

metrics_path = dataframes_dir / "ae_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics: {metrics_path}")

### Finish WandB

Closing any active WandB run.

In [ ]:
import wandb

if wandb.run is not None:
    wandb.finish()